# Data Analyst (итерация 2)

# Data Analyst Report — Fake Job Postings EDA

**Бизнес-задача:** Бинарная классификация мошеннических вакансий для HR-площадки. Цель — снизить нагрузку на ручную модерацию и защитить пользователей от скам-постингов.

**Метрика-приоритет для DS:** F1 / recall класса `fraudulent=1` при контролируемом precision.

**Что покажет этот EDA:**
1. Размерность и структуру очищенного датасета (num / cat / text колонки).
2. Степень дисбаланса классов `fraudulent` (ожидаем ~5% позитивов).
3. Корреляции числовых признаков с таргетом + heatmap.
4. Различия распределений топ-числовых признаков по классам.
5. Fraud-rate по топ-категориям `employment_type`, `required_experience`, `required_education`, `industry` и пр.
6. Сигнал из текстовых колонок: длина текстов и доля пустых значений по классам.

Все графики собираются в список `FIGS`, stdout-ы печатают конкретные числа для дальнейшего автоматического анализа.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

FIGS = []

DF = pd.read_csv('/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv')
TARGET = 'fraudulent'

print('Shape:', DF.shape)
print('\nDtypes:')
print(DF.dtypes)
print('\nHead:')
print(DF.head())

Shape: (17880, 38)

Dtypes:
title                                                       str
company_profile                                             str
description                                                 str
requirements                                                str
benefits                                                    str
telecommuting                                             int64
has_company_logo                                          int64
has_questions                                             int64
fraudulent                                                int64
employment_type_Contract                                  int64
employment_type_Full-time                                 int64
employment_type_Other                                     int64
employment_type_Part-time                                 int64
employment_type_Temporary                                 int64
required_experience_Associate                             int64
required_exp

## Обзор датасета

Классифицируем колонки на числовые / категориальные (low-/high-cardinality) / текстовые и печатаем базовую статистику.

In [ ]:
n_rows, n_cols = DF.shape
print(f'Rows: {n_rows}, Cols: {n_cols}')

# Классификация колонок
TEXT_CANDIDATES = ['title', 'company_profile', 'description', 'requirements', 'benefits']
TEXT_COLS = [c for c in TEXT_CANDIDATES if c in DF.columns]

num_cols_all = DF.select_dtypes(include=[np.number]).columns.tolist()
NUM_COLS = [c for c in num_cols_all if c != TARGET]

obj_cols = DF.select_dtypes(include=['object']).columns.tolist()
CAT_COLS = [c for c in obj_cols if c not in TEXT_COLS]

print(f'\nNUM columns ({len(NUM_COLS)}): {NUM_COLS}')
print(f'\nCAT columns ({len(CAT_COLS)}): {CAT_COLS}')
print(f'\nTEXT columns ({len(TEXT_COLS)}): {TEXT_COLS}')

print('\nDtype counts:')
print(DF.dtypes.value_counts())

print('\nNaN-rate per column (top-20):')
nan_rate = DF.isna().mean().sort_values(ascending=False)
print(nan_rate.head(20).round(4))

print('\nNumeric describe:')
print(DF[NUM_COLS + [TARGET]].describe().T.round(4))

print('\nCategorical cardinality:')
for c in CAT_COLS:
    print(f'  {c}: {DF[c].nunique()} unique')

Rows: 17880, Cols: 38
<string>:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.

NUM columns (32): ['telecommuting', 'has_company_logo', 'has_questions', 'employment_type_Contract', 'employment_type_Full-time', 'employment_type_Other', 'employment_type_Part-time', 'employment_type_Temporary', 'required_experience_Associate', 'required_experience_Director', 'required_experience_Entry level', 'required_experience_Executive', 'required_experience_Internship', 'required_experience_Mid-Senior level', 'required_experience_Not Applicable', 'required_education_Associate Degree', "r

## Распределение target `fraudulent`

Ожидаем сильный дисбаланс классов. Печатаем абсолютные и относительные частоты + imbalance ratio — это ключевой факт для выбора стратегии валидации и метрик.

In [ ]:
vc = DF[TARGET].value_counts().sort_index()
vc_norm = DF[TARGET].value_counts(normalize=True).sort_index()

print('Target value_counts:')
print(vc)
print('\nTarget proportions:')
print(vc_norm.round(5))

n_neg = int(vc.get(0, 0))
n_pos = int(vc.get(1, 0))
imbalance = n_neg / max(n_pos, 1)
pos_rate = n_pos / (n_neg + n_pos)
print(f'\nPositives (fraudulent=1): {n_pos}')
print(f'Negatives (fraudulent=0): {n_neg}')
print(f'Positive rate: {pos_rate:.4f} ({pos_rate*100:.2f}%)')
print(f'Imbalance ratio (neg:pos): {imbalance:.2f} : 1')

# Plot 1: bar chart
fig1 = px.bar(
    x=['Not fraudulent (0)', 'Fraudulent (1)'],
    y=[n_neg, n_pos],
    text=[n_neg, n_pos],
    title='Target distribution (counts)',
    labels={'x': 'fraudulent', 'y': 'count'},
    color=['Not fraudulent (0)', 'Fraudulent (1)'],
    color_discrete_sequence=['#3498db', '#e74c3c'],
)
fig1.update_traces(textposition='outside')
FIGS.append(fig1)

# Plot 2: pie chart
fig2 = px.pie(
    names=['Not fraudulent (0)', 'Fraudulent (1)'],
    values=[n_neg, n_pos],
    title='Target distribution (share)',
    color_discrete_sequence=['#3498db', '#e74c3c'],
    hole=0.35,
)
FIGS.append(fig2)

Target value_counts:
fraudulent
0    17014
1      866
Name: count, dtype: int64

Target proportions:
fraudulent
0    0.95157
1    0.04843
Name: proportion, dtype: float64

Positives (fraudulent=1): 866
Negatives (fraudulent=0): 17014
Positive rate: 0.0484 (4.84%)
Imbalance ratio (neg:pos): 19.65 : 1


## Числовые признаки vs target

Смотрим Pearson-корреляции всех числовых признаков с `fraudulent`, строим heatmap корреляций и сравниваем распределения топ-2 признаков по классам.

In [ ]:
if len(NUM_COLS) > 0:
    corr_full = DF[NUM_COLS + [TARGET]].corr()
    corr_target = corr_full[TARGET].drop(TARGET).sort_key(key=lambda s: s.abs()) if False else corr_full[TARGET].drop(TARGET)
    corr_target_sorted = corr_target.reindex(corr_target.abs().sort_values(ascending=False).index)

    print('Correlation with target (sorted by |corr|), top-10:')
    print(corr_target_sorted.head(10).round(4))
    print('\nCorrelation with target, bottom-5 (by |corr|):')
    print(corr_target_sorted.tail(5).round(4))

    # Plot 3: heatmap correlations
    fig3 = px.imshow(
        corr_full.round(3),
        text_auto=True,
        aspect='auto',
        color_continuous_scale='RdBu_r',
        zmin=-1, zmax=1,
        title='Correlation heatmap (numeric features + target)',
    )
    FIGS.append(fig3)

    # Plot 4: |corr| with target
    abs_corr = corr_target_sorted.abs()
    fig4 = px.bar(
        x=abs_corr.values,
        y=abs_corr.index,
        orientation='h',
        title='|Correlation| with target (fraudulent)',
        labels={'x': '|corr|', 'y': 'feature'},
        color=abs_corr.values,
        color_continuous_scale='Viridis',
    )
    fig4.update_layout(yaxis={'categoryorder': 'total ascending'})
    FIGS.append(fig4)

    # Plot 5: top-2 numeric features - distribution by target class
    top2 = corr_target_sorted.head(2).index.tolist()
    print(f'\nTop-2 numeric features by |corr|: {top2}')
    for feat in top2:
        g0 = DF.loc[DF[TARGET] == 0, feat]
        g1 = DF.loc[DF[TARGET] == 1, feat]
        print(f'  {feat}: mean(target=0)={g0.mean():.4f}, mean(target=1)={g1.mean():.4f}, '
              f'median(0)={g0.median():.4f}, median(1)={g1.median():.4f}')

    if len(top2) >= 1:
        feat = top2[0]
        fig5 = go.Figure()
        fig5.add_trace(go.Histogram(x=DF.loc[DF[TARGET]==0, feat], name='fraudulent=0',
                                    opacity=0.6, marker_color='#3498db', nbinsx=40, histnorm='probability density'))
        fig5.add_trace(go.Histogram(x=DF.loc[DF[TARGET]==1, feat], name='fraudulent=1',
                                    opacity=0.6, marker_color='#e74c3c', nbinsx=40, histnorm='probability density'))
        fig5.update_layout(barmode='overlay', title=f'Distribution of "{feat}" by target class',
                           xaxis_title=feat, yaxis_title='density')
        FIGS.append(fig5)
else:
    print('No numeric columns available besides target.')

Correlation with target (sorted by |corr|), top-10:
has_company_logo                                 -0.2620
required_education_Some High School Coursework    0.1254
has_questions                                    -0.0916
required_education_High School or equivalent      0.0563
required_education_Bachelor's Degree             -0.0540
required_experience_Associate                    -0.0539
employment_type_Part-time                         0.0447
location_freq                                    -0.0427
required_experience_Entry level                   0.0352
telecommuting                                     0.0345
Name: fraudulent, dtype: float64

Correlation with target, bottom-5 (by |corr|):
required_education_Vocational - Degree   -0.0041
required_experience_Director             -0.0033
required_education_Doctorate             -0.0018
required_education_Professional           0.0017
employment_type_Full-time                 0.0001
Name: fraudulent, dtype: float64

Top-2 numeric feat

## Категориальные признаки

Для наиболее информативных категориальных колонок смотрим **fraud-rate (mean target) по топ-10 категориям** — это прямой сигнал, где в данных концентрируется фрод.

In [ ]:
# Выбираем категориальные с разумной cardinality (>1 и <=1000) для анализа
cat_stats = []
for c in CAT_COLS:
    nu = DF[c].nunique(dropna=True)
    if nu > 1:
        cat_stats.append((c, nu))
cat_stats.sort(key=lambda x: x[1], reverse=True)
print('Categorical columns by cardinality:')
for c, nu in cat_stats:
    print(f'  {c}: {nu} unique')

# Берём топ-3 по cardinality (самые «богатые» сигналом), но не более 3
TOP_CAT = [c for c, _ in cat_stats[:3]]
print(f'\nTop-3 categorical features for deep-dive: {TOP_CAT}')

global_rate = DF[TARGET].mean()
print(f'Global fraud rate: {global_rate:.4f}')

for c in TOP_CAT:
    grp = DF.groupby(c)[TARGET].agg(['count', 'mean']).rename(columns={'mean': 'fraud_rate'})
    grp = grp[grp['count'] >= 20]  # убираем шумные редкие категории
    top10 = grp.sort_values('fraud_rate', ascending=False).head(10)
    print(f'\n=== {c}: top-10 categories by fraud_rate (count>=20) ===')
    print(top10.round(4))

    fig = px.bar(
        top10.reset_index(),
        x='fraud_rate', y=c, orientation='h',
        text='count',
        title=f'Fraud rate by {c} (top-10, min 20 samples; global={global_rate:.3f})',
        color='fraud_rate', color_continuous_scale='Reds',
    )
    fig.add_vline(x=global_rate, line_dash='dash', line_color='black',
                  annotation_text=f'global={global_rate:.3f}', annotation_position='top')
    fig.update_layout(yaxis={'categoryorder': 'total ascending'})
    FIGS.append(fig)

Categorical columns by cardinality:

Top-3 categorical features for deep-dive: []
Global fraud rate: 0.0484


## Текстовые признаки

Тексты (`title`, `description`, ...) — **самый сильный сигнал** в задачах фрод-детекции. Смотрим:
- **word_count** по каждой текстовой колонке и её среднее по классам,
- **NaN/empty-rate** по классам (подозрительные вакансии часто имеют пустые company_profile / requirements / benefits).

In [ ]:
def _wc(s):
    if not isinstance(s, str):
        return 0
    return len(s.split())

print(f'Text columns: {TEXT_COLS}')

wc_stats = {}
empty_stats = {}
for c in TEXT_COLS:
    wc = DF[c].apply(_wc)
    wc_stats[c] = wc
    empty_mask = DF[c].isna() | (DF[c].astype(str).str.strip() == '')
    empty_stats[c] = empty_mask

    mean_wc_0 = wc[DF[TARGET] == 0].mean()
    mean_wc_1 = wc[DF[TARGET] == 1].mean()
    med_wc_0 = wc[DF[TARGET] == 0].median()
    med_wc_1 = wc[DF[TARGET] == 1].median()
    empty_0 = empty_mask[DF[TARGET] == 0].mean()
    empty_1 = empty_mask[DF[TARGET] == 1].mean()

    print(f'\n--- {c} ---')
    print(f'  mean word_count: target=0 -> {mean_wc_0:.2f}, target=1 -> {mean_wc_1:.2f}')
    print(f'  median word_count: target=0 -> {med_wc_0:.1f}, target=1 -> {med_wc_1:.1f}')
    print(f'  empty/NaN rate: target=0 -> {empty_0:.4f}, target=1 -> {empty_1:.4f}')

# Plot 9: distribution длины текста по target для самой «толстой» колонки
if TEXT_COLS:
    # берём колонку с наибольшим средним word_count — обычно description
    pick = max(TEXT_COLS, key=lambda c: wc_stats[c].mean())
    print(f'\nChosen text column for length distribution plot: {pick}')
    wc = wc_stats[pick]
    # обрезаем хвост на 99-м перцентиле только для визуализации
    cap = np.percentile(wc, 99)
    fig9 = go.Figure()
    fig9.add_trace(go.Histogram(x=wc[DF[TARGET]==0].clip(upper=cap), name='fraudulent=0',
                                opacity=0.6, marker_color='#3498db', nbinsx=50, histnorm='probability density'))
    fig9.add_trace(go.Histogram(x=wc[DF[TARGET]==1].clip(upper=cap), name='fraudulent=1',
                                opacity=0.6, marker_color='#e74c3c', nbinsx=50, histnorm='probability density'))
    fig9.update_layout(barmode='overlay',
                       title=f'Word count distribution of "{pick}" by target (clipped at p99={cap:.0f})',
                       xaxis_title='word_count', yaxis_title='density')
    FIGS.append(fig9)

# Plot 10: bar chart empty-rate по target для каждой text-колонки
if TEXT_COLS:
    rows = []
    for c in TEXT_COLS:
        m = empty_stats[c]
        rows.append({'column': c, 'target': 'fraudulent=0', 'empty_rate': m[DF[TARGET]==0].mean()})
        rows.append({'column': c, 'target': 'fraudulent=1', 'empty_rate': m[DF[TARGET]==1].mean()})
    df_empty = pd.DataFrame(rows)
    fig10 = px.bar(
        df_empty, x='column', y='empty_rate', color='target', barmode='group',
        title='Empty/NaN rate of text columns by target class',
        color_discrete_map={'fraudulent=0': '#3498db', 'fraudulent=1': '#e74c3c'},
        text=df_empty['empty_rate'].round(3),
    )
    fig10.update_traces(textposition='outside')
    FIGS.append(fig10)

print(f'\nTotal figures collected: {len(FIGS)}')

Text columns: ['title', 'company_profile', 'description', 'requirements', 'benefits']

--- title ---
  mean word_count: target=0 -> 3.75, target=1 -> 4.02
  median word_count: target=0 -> 3.0, target=1 -> 3.0
  empty/NaN rate: target=0 -> 0.0000, target=1 -> 0.0000

--- company_profile ---
  mean word_count: target=0 -> 95.65, target=1 -> 31.71
  median word_count: target=0 -> 86.0, target=1 -> 0.0
  empty/NaN rate: target=0 -> 0.1599, target=1 -> 0.6778

--- description ---
  mean word_count: target=0 -> 171.04, target=1 -> 158.75
  median word_count: target=0 -> 147.0, target=1 -> 113.5
  empty/NaN rate: target=0 -> 0.0000, target=1 -> 0.0023

--- requirements ---
  mean word_count: target=0 -> 79.03, target=1 -> 58.41
  median word_count: target=0 -> 63.0, target=1 -> 34.0
  empty/NaN rate: target=0 -> 0.1495, target=1 -> 0.1778

--- benefits ---
  mean word_count: target=0 -> 30.02, target=1 -> 29.45
  median word_count: target=0 -> 6.0, target=1 -> 5.0
  empty/NaN rate: target=0 -

## Что покажет EDA (структура выводов)

1. **Масштаб и структура данных.** Размерность очищенного CSV, разбивка колонок на num / cat / text, NaN-rate после чистки DE.
2. **Дисбаланс target.** Позитивный класс — редкий (ожидается ~5%). Это диктует выбор метрик (F1/recall positive) и стратегию CV (stratified) для DS.
3. **Числовые сигналы.** Корреляции `telecommuting`, `has_company_logo`, `has_questions` и `_freq`-кодированных колонок с `fraudulent` — первый быстрый срез. Heatmap плюс ранжированный бар `|corr|` дают компактное представление силы сигналов.
4. **Различия распределений.** Для топ-2 числовых признаков сравнение распределений по классам подсказывает, линейно ли разделим сигнал или нужны нелинейности/взаимодействия.
5. **Категориальные признаки.** Fraud-rate по топ-10 категориям в топ-3 cat-колонках показывает, где концентрируется мошенничество (напр., определённые `industry`/`employment_type`/`required_experience`). Сравнение с global fraud-rate (пунктирная линия на графиках) сразу даёт DS гипотезы для фичей / правил.
6. **Текст.** Статистика word_count и доли пустых текстовых полей по классам — ключевая гипотеза: фродовые вакансии часто короче/с пустым company_profile. Это мотивирует DS делать текстовые фичи (TF-IDF, длины, флаги пустоты) и, вероятно, даст самый сильный lift.

**Следующие шаги для DS:**
- Stratified CV по `fraudulent`.
- Текстовые фичи (TF-IDF / embeddings) + числовые/категориальные через gradient boosting.
- Оптимизация порога под целевой precision; отчёт по recall@precision.
- Эксперимент с class_weight / focal loss вместо слепого oversampling.